# Тестване

Осигуряването и управлението на качеството е важен аспект от разработката на софтуер. За целта се използват различни методи, като т.нар. тестове главно се делят на end-to-end, integration и unit тестове. За първите две отговорността обикновено е у QA инженера, докато unit тестовете обикновено се пишат от разработчиците, понеже са white-box тестове (т.е. тестващия трябва да има знание всеки компонент вътрешно как изглежда). Това разделение е условно, всеки който има достатъчно знания може да пише всякакъв вид тестове!

![различни видове тестове](assets/tests.png)

Целта на unit test-овете е да проиграят всички възможни ключови ситуации, свързани с **един конкретен компонент** и да проверят дали този компонент се държи според критериите закодирани вътре в теста. Тук под компонент се визира клас или метод изпълняван в изолация от останалия свят! Разликата при integration, end-to-end и други тестове е че там има компоненти от страничния свят: мрежа, файлова система, база данни, взаимодействие с друг компонент и т.н. Повече по темата може да намерите в книгата [The Art of Unit Testing](https://www.artofunittesting.com/).

## Before

В папката `13 - Modules` има пример за тестване на един компонент, макар и примитивен такъв. Намира се в `game.engine` модула:

```python
if __name__ == "__main__":
    # Executed when running `python3 -m game.engine`
    
    from game.players.mock_player import MockPlayer

    print("Testing win case...")
    player_win = MockPlayer(1, "oba")
    engine_win = BesenitsaEngine("foobar", player_win)

    assert engine_win.guess() == GameState.ONGOING
    assert engine_win.guess() == GameState.ONGOING
    assert engine_win.guess() == GameState.WON
    print("Test OK.")

    print("Testing lose case...")
    player_lose = MockPlayer(3, "asdg")
    engine_lose = BesenitsaEngine("foobar", player_lose)

    assert engine_lose.guess() == GameState.ONGOING
    assert engine_lose.guess() == GameState.ONGOING
    assert engine_lose.guess() == GameState.ONGOING
    assert engine_lose.guess() == GameState.LOST
    print("Test OK.")
```

Тестването по този начин, освен че може да бъде по-трудоемко и по-трудно за проследяване и изпълнение за по-сложни ситуации и проекти, има и друг недостатък, че когато нещо не е както трябва (някой `assert` не минава), нямаме детайлна информация за това кое не е правилно. Например, ако `engine_win.guess()` връща `GameState.LOST` вместо `GameState.WON`, ще получим само `AssertionError`, без да знаем какъв е точно върнатия резултат на `engine_win.guess()` и защо не е равен на `GameState.WON`.

Още повече, най-главния недостатък на този начин на "тестване" е че така се изпълняват последователно тестовете и ако един гръмне, то другите няма да се изпълнят. А ние не искаме това да е така - трябва всеки тест да се изпълнява независимо от другите - без споделена памет и без споделено състояние и т.н.

**Забележка:** изпълнението на тестове в отделни нишки звучи примамливо, но в практиката не винаги е добра идея и често може да бъде лош сигнал за по-опитните QA инженери. Повечето test runner-и в Python изпълняват всичко последователно.

## Основни компоненти в тестовата екосистема на Python

* Discovery & runner - откриват какви тестове има и ги изпълняват. Обикновено използват конвенции за именоване или наследяване
* Framework - задава структурата/формата за писане на тестовете. Определя формата и обхвата на видимост на setUp/tearDown
* Assertions library - какви проверки можем да правим на готово

В Python тези компоненти имат силна свързаност по между си и обикновено се наричат с названието "testing framework". В други езици имат слаба свързаност и за всеки компонент съществуват много опции.

## After (a.k. unit testing frameworks in Python)

Съществуват няколко основни test runner-а в Python света, като `unittest`, `pytest`, `nose`, `nose2` и други. Ще разгледаме двата най-използвани - `unittest` и `pytest`.

### `unittest`

`unittest` е ***вградена*** библиотека (от Python 2.1 насам), която ни предоставя както framework, така и runner за тестове.

#### Особености

* Всеки тестови случай е метод на клас, наследяващ `unittest.TestCase`
* Използват се `assert...` методи на класа вместо `assert` ключовата дума
* Трябва всеки тестови модул да извика `unittest.main()` ако искаме той да може да бъде изпълнен директно като Python-ска програма

Ако трябва горните два test case-a да ги пренапишем с `unittest`, ще изглеждат по този начин:

```python
import unittest

from tests.mocks.mock_player import MockPlayer  # moved inside the tests/ package
from game.engine import BesenitsaEngine, GameState

class EngineTests(unittest.TestCase):
    def test_foobar_win(self):
        player_win = MockPlayer(1, "oba")
        engine_win = BesenitsaEngine("foobar", player_win)

        self.assertEqual(engine_win.guess(), GameState.ONGOING)
        self.assertEqual(engine_win.guess(), GameState.ONGOING)
        self.assertEqual(engine_win.guess(), GameState.WON)

    def test_foobar_lose(self):
        player_lose = MockPlayer(3, "asdg")
        engine_lose = BesenitsaEngine("foobar", player_lose)

        self.assertEqual(engine_lose.guess(), GameState.ONGOING)
        self.assertEqual(engine_lose.guess(), GameState.ONGOING)
        self.assertEqual(engine_lose.guess(), GameState.ONGOING)
        self.assertEqual(engine_lose.guess(), GameState.LOST)

if __name__ == '__main__':
    unittest.main()
```


`self.assertEqual` е един от многото методи, които предоставя `unittest` за проверка на различни условия. Пълен списък може да намерите [тук](https://docs.python.org/3/library/unittest.html#assert-methods). Те са:

* `assertEqual(a, b)` - проверява дали `a == b`
* `assertTrue(x)` - проверява дали `bool(x) is True`
* `assertFalse(x)` - проверява дали `bool(x) is False`
* `assertIs(a, b)` - проверява дали `a is b`
* `assertIsNot(a, b)` - проверява дали `a is not b`
* `assertIsNone(x)` - проверява дали `x is None`
* `assertIsNotNone(x)` - проверява дали `x is not None`
* `assertIn(a, b)` - проверява дали `a in b`
* `assertNotIn(a, b)` - проверява дали `a not in b`
* `assertIsInstance(a, b)` - проверява дали `isinstance(a, b)`
* `assertNotIsInstance(a, b)` - проверява дали `not isinstance(a, b)`

Ползата им можем да видим например ако променим във `test_foobar_win` да очакваме накрая `GameState.LOST` вместо `GameState.WON` (или по някакъв друг начин променим кода така, че да имаме грешно поведение спрямо тестовете). Това output-ът от този тест ще е достатъчно информативен (сравнението показва всички разлики между двата обекта):

```
======================================================================
FAIL: test_foobar_win (test_engine.EngineTests)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/alexander.ignatov/Desktop/besenitsa/tests/test_engine.py", line 25, in test_foobar_win
    self.assertEqual(engine_win.guess(), GameState.LOST)
AssertionError: <GameState.WON: 1> != <GameState.LOST: 2>
```

*Интересен факт:* специализиран test framework, като например този в Django, наследява `unittest` и
добавя свои собствени assert методи, виж [тук](https://docs.djangoproject.com/en/6.1/topics/testing/tools/#simpletestcase).

#### Команди

Директно изпълнение на един файл с тестове:

In [1]:
!python3 test_single_simple_unittest.py

.
----------------------------------------------------------------------
Ran 1 test in 0.000s

OK


Изпълнение на един модул с тестове:

*Важно*: ⚠️ При достъп през пакет (т.е. с точка, както по-долу имаме пакетът `tests`, в който се намира `test_engine` модула) трябва пакетът да съдържа `__init__.py` файл, макар и празен (това показва, че пакетът **не е** само т.нар. **namespace** package).

In [2]:
!python3 -m unittest tests.test_engine

.....
----------------------------------------------------------------------
Ran 5 tests in 0.000s

OK


Автоматично откриване на всички тестове в *текущата* директория:

(търси всички python файлове, започващи с `test`, и ги изпълнява)

In [3]:
!python3 -m unittest discover

.......
----------------------------------------------------------------------
Ran 7 tests in 0.000s

OK


Автоматично откриване на всички тестове в *дадена* директория (в случая `tests`):

In [4]:
!python3 -m unittest discover -s tests

......
----------------------------------------------------------------------
Ran 6 tests in 0.000s

OK


Изпълнение на всички файлове с име, започващо с  `test_`, в директория, наречена `tests`:

In [5]:
!python3 -m unittest discover -s tests -p "test_*.py"

......
----------------------------------------------------------------------
Ran 6 tests in 0.000s

OK


Подробен output се дава с добавяне на параметъра `-v`:

In [6]:
!python3 -m unittest discover -s tests -v

test_firstGuess_isE (test_ai_player.TestAIPlayer) ... ok
test_cat_lose (test_engine.EngineTests) ... ok
test_cat_win (test_engine.EngineTests) ... ok
test_foobar_lose (test_engine.EngineTests) ... ok
test_foobar_win (test_engine.EngineTests) ... ok
test_initialWord_isMaskedCorrectly (test_engine.EngineTests) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.000s

OK


Ако кодът не се намира в корена на директорията, а например в папка, наречена `src`, можем да добавим параметър `-t src`, за да кажем на `unittest` да изпълни тестовете оттам:

```bash
python3 -m unittest discover -s tests -t src
```

### `pytest`

#### Особености

`pytest` **не е** вградена библиотека, но има някои предимства пред `unittest`, като:

* по-малко boilerplate (не се наследява от `unittest.TestCase`, използва се `assert` ключовата дума, няма main entry point)
* възможност за филтриране на тестове
* екосистема от стотици плъгини за разширяване на функционалността

Хубаво е да се отбележи, че тестове, написани с `unittest` са **съвместими** с `pytest` и могат да се изпълняват с него. Обратното, обаче, не е вярно.

Ако трябва да пренапишем двата теста от по-горе за `pytest`, файлът би изглеждал така:

```python

from tests.mocks.mock_player import MockPlayer
from game.engine import BesenitsaEngine, GameState

def test_foobar_win():
    player_win = MockPlayer(1, "oba")
    engine_win = BesenitsaEngine("foobar", player_win)

    assert engine_win.guess() == GameState.ONGOING
    assert engine_win.guess() == GameState.ONGOING
    assert engine_win.guess() == GameState.WON

def test_foobar_lose():
    player_lose = MockPlayer(3, "asdg")
    engine_lose = BesenitsaEngine("foobar", player_lose)

    assert engine_lose.guess() == GameState.ONGOING
    assert engine_lose.guess() == GameState.ONGOING
    assert engine_lose.guess() == GameState.ONGOING
    assert engine_lose.guess() == GameState.LOST

#### Команди

Преди всичко, трябва да инсталираме `pytest` чрез PIP например:

In [7]:
!pip install pytest


[notice] A new release of pip available: 22.2.2 -> 22.3.1
[notice] To update, run: pip install --upgrade pip


Това ни предоставя и командата `pytest`:

In [8]:
!pytest --version

pytest 7.2.0


Извикването на командата `pytest` е почти еквивалентно на това да се изпълни модула чрез `python3 -m pytest`. Единствената разлика е, че втория начин ще добави и текущата директория към `sys.path`.

Директно изпълнение на един файл с тестове:

In [9]:
!pytest test_single_simple_pytest.py

============================= test session starts ==============================
platform darwin -- Python 3.10.8, pytest-7.2.0, pluggy-1.0.0
rootdir: /Users/alexander.ignatov/Documents/PythonCourse2022/15 - Testing
collected 1 item                                                               

test_single_simple_pytest.py .                                           [100%]

============================== 1 passed in 0.00s ===============================


Изпълнение на всички тестове в директория `tests`:

In [10]:
!pytest tests

============================= test session starts ==============================
platform darwin -- Python 3.10.8, pytest-7.2.0, pluggy-1.0.0
rootdir: /Users/alexander.ignatov/Documents/PythonCourse2022/15 - Testing
collected 6 items                                                              

tests/test_ai_player.py .                                                [ 16%]
tests/test_engine.py .....                                               [100%]

============================== 6 passed in 0.01s ===============================


Изпълнение само на тестове, отговарящи на даден критерий (в случая: всички от `EngineTests` класа, които нямат `foobar` в името си):

In [11]:
!pytest -k "EngineTests and not foobar"

============================= test session starts ==============================
platform darwin -- Python 3.10.8, pytest-7.2.0, pluggy-1.0.0
rootdir: /Users/alexander.ignatov/Documents/PythonCourse2022/15 - Testing
collected 8 items / 5 deselected / 3 selected                                  

tests/test_engine.py ...                                                 [100%]

======================= 3 passed, 5 deselected in 0.01s ========================


Изпълнение по Node ID:

(на всеки открит тест от `pytest` бива сложено Node ID, което се съставя от името на файла, последвано от специфичния "път", който води до scope-а на теста, т.е. имена на класове, методи/функции и евентуални техни параметри, разделени от `::`)

In [12]:
!pytest tests/test_engine.py::EngineTests

============================= test session starts ==============================
platform darwin -- Python 3.10.8, pytest-7.2.0, pluggy-1.0.0
rootdir: /Users/alexander.ignatov/Documents/PythonCourse2022/15 - Testing
collected 5 items                                                              

tests/test_engine.py .....                                               [100%]

============================== 5 passed in 0.01s ===============================


In [13]:
!pytest tests/test_engine.py::EngineTests::test_initialWord_isMaskedCorrectly

============================= test session starts ==============================
platform darwin -- Python 3.10.8, pytest-7.2.0, pluggy-1.0.0
rootdir: /Users/alexander.ignatov/Documents/PythonCourse2022/15 - Testing
collected 1 item                                                               

tests/test_engine.py .                                                   [100%]

============================== 1 passed in 0.00s ===============================


По-подробна употреба, като изпълнение на тестове по маркери, такива от пакети и т.н. може да видите в [документацията на pytest](https://docs.pytest.org/en/7.1.x/how-to/usage.html).

## Добри практики

### Файлова структура

Модулът/пакетът с тестови код трябва да е отделен от пакетите, съдържащи основния код. Това е за да се избегне циклична зависимост между тях (зависимостта трябва да е в едната посока само - тестовете зависят от основния код, но той от тях - не). Това означава, че трябва да има две отделни директории - една с основния код, друга с тестовете:

In [14]:
!tree game tests

game
├── engine.py
├── level.py
├── player.py
└── players
    ├── ai.py
    └── input_player.py
tests
├── __init__.py
├── mocks
│   ├── __init__.py
│   └── mock_player.py
├── test_ai_player.py
└── test_engine.py

1 directory, 5 files


При src-базирани структури на проекта, това ще изглежда така:

```
src
└── game
    ├── engine.py
    ├── level.py
    ├── player.py
    └── players
        ├── ai.py
        └── input_player.py
tests
├── __init__.py
├── mocks
│   ├── __init__.py
│   └── mock_player.py
├── test_ai_player.py
└── test_engine.py
```

В случай, че имаме и unit, и integration тестове например, е добре да ги разделим (за да може лесно да изпълняваме само unit тестовете например):

```
tests/
    ├── unit/
    |   ├── __init__.py
    |   ├── test_component1.py
    |   └── test_component2.py
    └── integration/
        ├── __init__.py
        └── test_components_integration.py
```

### Конвенции за име

Хубаво е всички файлове и функции/методи с тестови код да започват името си с `test`, за да бъдат автоматично разпознавани от всички test runner-и. Файловете трябва да съдържат името на компонента (или нещо, което не е двусмислено и подсказва кой е въпросния компонент), както и евентуално и тази част от функционалността, която се тества (в случай, че повече от един файл се занимават с един и същ компонент). В нашия случай `test_engine.py` или `test_BesenitsaEngine.py` биха били добри имена.

Името на един тест e хубаво да подсказва какво се тества, какво се очаква и евентуални при какви специфични условия (например `test_initial_word_is_masked_correctly` или разделено семантично чрез комбинация от snake_case и camelCase като `test_initialWord_isMaskedCorrectly`). Съществува практика и за номерирането на тестовете (т.е. `test_001_initialWord_isMaskedCorrectly`). Силно препоръчително е всеки тест да си има docstring с описание на тестовия сценарий. Имайте предвид, че текстовото описание и кодът може да започнат да се разминават във времето, но каноничната форма е кодът! Ако трябва да избирате между двете винаги е по-добре да имаме експлицитен код и имена на класове/методи/променливи!

Имената на допълнителните класове/функции, създадени специално за тестовете, е хубаво да подсказват каква е ролята им в тестовете - дали са fixtures, helpers, stubs, mocks, spies и т.н. **Забележка:** с тези термини се назоват помощни функции като разликите между тях са семантични и в практиката понякога е трудно да проделим дали нещо е stub, mock или spy. За повече информация прочетете [Test double в Wikipedia](https://en.wikipedia.org/wiki/Test_double).

### Arrange-Act-Assert

Тази схема ни помага да структурираме по-нагледно съдържанието на един тест. Тя ни казва да разделим кода в един тестови случай на три последователни части:

* Arrange - инстанциране на обекта, който ще се тества, подготовка на данните (fixtures), нагласяне на mock-ове, инжектиране на зависимости (dependency injection) и т.н.
* Act - действието, което всъщност тестваме (най-често представлява просто извикване на метода/методите от обекта, който/които ще се тества/тестват)
* Assert - проверки

Например в `EngineTests` имаме следните тестове:

```python
# ...

class EngineTests(unittest.TestCase):
    def test_initialWord_isMaskedCorrectly(self):
        # Arrange
        player = MockPlayer(1, "a")
        sut = BesenitsaEngine("cat", player)
        expected = "C_T"

        # Act
        result = sut.masked_word

        # Assert
        self.assertEqual(result, expected)
    
    def test_cat_win(self):
        # Arrange
        player_win = MockPlayer(1, "a")
        sut = BesenitsaEngine("cat", player_win)

        # Act
        result = sut.guess()

        # Assert
        self.assertEqual(result, GameState.WON)

    def test_cat_lose(self):
        # Arrange
        player_win = MockPlayer(1, "b")
        sut = BesenitsaEngine("cat", player_win)

        # Act
        result = sut.guess()

        # Assert
        self.assertEqual(result, GameState.LOST)
    
    # ...
```


(`sut` идва от `System Under Test` - това е обекта, който се тества в текущия unit test)

## Изпълнение от IDE/editors

### PyCharm

1. В Project tool избираме тестовата директория
2. В контекстното ѝ меню избираме съответната run команда.

![pycharm](assets/pycharm.png)

Повече инфо (в сайта на PyCharm)[https://www.jetbrains.com/help/pycharm/performing-tests.html].

### VS Code

Плъгина [Python](https://marketplace.visualstudio.com/items?itemName=ms-python.python) има команди свързани с тестовете, като "Debug all unit tests", "Run all unit tests", и т.н.:

![vscode](assets/vscode-python.png)

При първото изпълнение на такава команда VS Code ще помогне да се настроят накои параметри за изпълнението на тестовете (unittest vs pytest, директория, т.н.), които ще запамети като настройки.

След това резултатът от тестовете ще се показва долу вляво в информационната лента:

![vscode status](assets/vscode-bottom.png)

Съществуват и плъгините [Text Explorer UI](https://marketplace.visualstudio.com/items?itemName=hbenl.vscode-test-explorer) и [Python Test Explorer](https://marketplace.visualstudio.com/items?itemName=LittleFoxTeam.vscode-python-test-adapter), които показва тестовете в ново меню (в лентата, която обикновено седи в лявата част на екрана):

![vscode test ui](assets/vscode-testui.png)

Както се вижда, добавя UI за изпълнението или дебъгването на конкретен тест или цял набор от тестове. Покачва до дефиницията на всеки тест дали последно е минал успешно или не, и ако не - показва грешката, която е хвърлил.

## Автоматизираното тестване на практика

Моля различавайте термините "**тестване**" и "**тестова автоматизация**" защото те имат различни цели!

Целта на **тестването** е откриване на нова информация за обекта на тестване: неочаквано поведение и/или бъгове, нов начин за използване, информация относно бързодействие, сигурност (или липсата на такива) и още и още. Чрез различни подходи, техники и инструменти се опитваме да набавим допълнителна информация, повече от това, което знаем до сега! Тук интересните факти се намират извън зоната на комфорта!

![целта на тестването](assets/the-purpose-of-testing.png)

Целта на **тестовата автоматизация** е закодиране на определено поведение/факти, т.ч. те да бъдат проверявани повторно. В книгите за тестване се използва понятието "оракул", без значение дали става дума за документ на човешки език или програмен код. Има се предвид това, което ни казва каноничната истина! Използването на повтаряеми тестове/техники/подходи обикновено достига максимум, след който рядко носи допълнителна информация. Това се нарича "Pesticide paradox".

В реалния живот на един софтуерен продукт имаме много повече от самия код, който сме написали: има DevOps скриптове, различни тестови и продукционни среди, бази данни, компоненти и системи на 3ти страни с които взаимодействаме, артефакти на софтуера (примерно компилирани билдове, докер контейнер, Линукс пакети), API токени и пароли и много много други. Това означава, че източниците на грешки и проблеми се намират на всички тези места съответно тестването и проверките трябва да се случват на всички тези места. Понякога един и същи език и тестови framework може да бъде достатъчен, за да закодираме проверки в среди и компоненти, които на пръв поглед не са свързани. В други случаи ще се наложи да използваме различни интрументи и програмни езици, за да си свършим работата.

![кръговрата на тестването](assets/testing-infinity.png)


В практиката писането на автоматизирани тестове обикновено не е първото нещо с което един инженер започва да се занимава. Доста често дори е последното. И редовно се случва, че всеки пише какъвто и да е вид тестове според както се случи и според както може.

![тестовата пирамида на практика](assets/testing-pyramid-in-practice.png)




### Coverage метрики и какво ни казват те

Най-често срещаната метрика за оценка на тестовете е т.нар. code coverage. Обикновено се има предвид или line coverage или branch coverage!
Тази метрика измерва броя редове или разклонения в една програма, които са били покрити/посетени/"изпълнени" за една работна сесия на програмата!
Можем да събираме coverage метрики както на ръка (примерно изпълняваме програмата от командния ред или през GUI), така и от изпълнения на автоматизирани тестове.

Важно е да се отбележи, че в Python при изчисления с условни изрази се случва т.нар. lazy evaluation и повечето инструменти за езика не го пресмятат като отделна метрика. Например в кода:

```
if user.is_admin and ....:
    pass
```

ако нашата програма винаги е изпълнена с потребител администратор, тогава coverage метриката за този ред винаги ще бъде една и съща защото интерпретатора на Python не взима 2то (или 3то) условие под внимание. Т.е. ако разчитаме само на line coverage метриката няма да можем да открием фактът, че програмата се изпълнява винаги с потребител администратор.

**Важно:** в практиката битува разбирането, че line coverage е метрика, която показва колко добри са нашите автоматизирани тестове и често се цели тя да бъде колкото се може по-висока! **Това разбиране е неправилно и подвеждащо! line coverage има отношение само към това, каква част от програмата е била изпълнена и сама по себе си няма никакво отношение към тестовете или тяхното качество!** Конкретно line coverage казва:

```
- изпълних една конкретна част от програмата (по един определен начин, с едни определени данни)
- в тази част от програмата нищо катастрофално не се обърка докато я изпълнявах
- т.е. нямаше синтактични грешки, нямаше изключения (например делене на нула или несъществуващ файл)
```

*Анти пример от реално iOS 9 приложение:* 2016 година, XCUITest/XCTest, navigation style smoke test има automation: натискаме бутони, движение по екрана; “100%” coverage, но едни и същи бъгове се появяваха редовно. Причината: 0 assertions, т.е. никакви конкретни проверки на поведение и странични ефекти от изпълнението на програмата. Единствения вариант при който този test suite връща FAILED е ако липсват UI елементи, ако техните локатори/имена са променени или ако програмата крашне катастрофално!

*Друг пример*: в [1](http://atodorov.org/blog/2016/12/27/mutation-testing-vs-coverage/) и [2](http://atodorov.org/blog/2017/04/05/mutation-testing-vs-coverage-pt-2/) 
са описани няколко бъга, които инструментите не засичат при положение че имаме 100% line/branch/mutation coverage. Примерите са на Python. В блога са описани и липсващите интеграционни тестове.

Едно от светилата в тестерската професия, Cem Kaner, [описва](https://www.kaner.com/pdfs/negligence_and_testing_coverage.pdf) повече от 100 различни coverage метрики. На практика се ползват само няколко от тях и инженерите се заблуждават, че "всичко е наред"!

*Лирическо отклонение:* в цитата от Cem Kaner, от горната презентация:
```
For example, 100% line coverage doesn’t just mean that you’ve executed every line of code; it also means that you’ve tested for every bug that can be revealed by simple execution of a line of code!
```
 
 ние хората редовно изчитаме само първата половинка на изречението "100% line coverage ... means that you’ve tested for every bug" докато същинското значение се намира във втората част на това изречение: **"every bug that can be revealed by simple execution of a line of code"**! 


 **ВАЖНО:** силата на вашите автоматизирани тестове идва от 1) какви тестови сценарии / поведение симулират те и 2) какви и колко проверки извършват тези тестове!

## Един зелен тест в ерата на ИИ

ИИ / LLM модели се справят сравнително добре с това да генерират код за тестова автоматизация без значение дали говорим за unit, integration или какъвто и да било друг вид тестове. Тук обаче изникват следните въпроси:

* Какво точно се проверява или кой е subject under test
* Колко точно се проверява, пр. само записи в БД или очаквани мрежови транзакции или и двете
* Доколко е адекватен един генериран тестови сценарии (и неговата конкретна реализация)
* Колко точно можем да му вярваме
* Кога знаем, че сме готови (aka какъв е нашият Definition of Done) - преди merge или преди release или преди deploy и т.н.
* Какво означава когато всичко е в зелено

![зелен тест със синтактична грешка](assets/green-test-syntax-error.png)

Отоворите на тези въпроси са специфични за конкретния проект, по който се работи, за конкретния екип и среда в която се работи. Препоръчително е обаче да имаме някаква дефиниция за тях (без значение каква). Когато се окаже, че дефиницията куца, напр. изпуснали сме елементарен бъг, можем да я допълним, т.ч. на следващата итерация да имаме по-строги критерии. Всичко това е "според зависи" и е трудно да се даде еднозначен отговор на горните въпроси, който отговор да бъде валиден в множество различни ситуации.

Един зелен тест, подобно на coverage метриката, не казва *"всичко е наред"*, а казва *"ето това конкретно нещо работи така както за сега сме описали, че трябва да работи"*. 

## Практически съвети как да мислим като тестер

При изследване на една програма/компонент/функционалност от гледната точка на откриване на нова интересна информация за тях можем да се оглеждаме за следните неща:

* Какво и как да счупим като въвеждаме странни входни данни: отрицателни числа, запетайки, букви на кирилица например, емоджита, файлове с дължина 0 или с много голяма дължина (примерно 10 гигабайта) и всякакви такива странни неща според вида на софтуера
* Да съпоставяме поведение както е написано в код срещу дефиницията както е написана като документи или коментари или имена в кода
* Да търсим несъответствия и неясноти - ако на мен не ми е ясно, вероятно и на този който го е правил не му е било ясно или имаме over engineering (примерно Claude Code)
* Да следим за странични ефекти: какво се записва в БД или на файловата система, дали имаме външни среди/услуги - примерно дали се вика някакво API или 3rd party система, която по някакъв начин консумира разултатите от компонента който се тества
* Места на които често се откриват несъответствия: get/set методи; тестове и код; компоненти; GUI vs cli vs API (предполага се, че всичките трябва да предоставят еднаква функционалност или под-множество); нива на абстракция между компоненти/апи-та - core vs. wrapper vs. add-on или едно и също АПИ изкарано през различни протоколи: JSON-RPC, REST, MCP и т.н.


При писане на автоматизирани тестови сценарии е добре да се стремим да покрием следното:

* Минимум 1 позитивен сценарии, който да покрива основното използване на функционалността така както е дефинирана по изискване: примерно: функцията sum(a, b) трябва да върне сумата на 2те числа a + b, проверяваме с 1+2, 5+3, за по правилен подход виж [Boundary-value analysis](https://en.wikipedia.org/wiki/Boundary-value_analysis). Добре е да избягваме еднакви входни данни, т.е. повече разнообразие може да засече по-интересни случки/бъгове
* Да са покрити всички негативни сценарии - обикновено това са всички извиквания с невалидни входни данни: float, string, bool, отрицателни числа - очаква се че ще се случи изключение и тестовете трябва да проверяват за него
* Проверка на всички останали състояния които връщат/хвърлят грешка и задължително проверка(assert) че резултатът/грешката е точно такава каквато си мислим ("грешката е вярна") - примерно проверка за ZeroDivisionError вместо RuntimeError!
* Минимум 1 интеграционен тест, т.е. викаме компонента/функцията отвън, по възможно най-близък начин както ще се изполва в production. Обикновено това е положителен сценарий. 
* По възможност да се стремим да имаме максимум на брой проверки, assert-ти на всичко което се случва в компонента, който се тества. Пр. записите в БД са станали +1, новия запис е с очакваните стойности така както са въведени, функцията е върнала резултат (return value), дали има/няма някакви генерирани събития и т.н.

Добър източник на идеи какво (и понякога как) да проверяваме са т.нар. странични ефекти на функците. 

*За уточнене:* чиста функция (pure function) е такава която не пише в паметта, не променя стойности на глобални променливи, не пише в БД или не вика нещо по мрежата. Всичко останало в реалния живот има странични ефекти, което води до промяна на вътрешно състояние на софтуерен компонент, очаквани междинни данни и др., което от своя страна във времето води до разминавки и интересни бъгове.

Обикновенно един набор от атоматизирани тестове (дори и за една и съща функционалност) е много по-голям като обем спрямо обемът код на това, което се тества. Добра практика е да се мисли/спазва принципът separation of concerns. Т.е. винаги да е ясно кой е subject under test, по възможност да взаимодействаме само с него и да провярваме само неговото състояние и страничните ефекти директно свързани с SUT. Останалите проверки са в други подобни тестови сценарии, които са 1% по различни. За съжаление това води до много повтаряем код и е сравнително тромаво за проследяване на ръка - къде свършват unit тестове, къде почват API тестове и т.н. Ако сте начинаещ програмист мислете за този принцип в момента на първоначалното писане на тестовете и след това го сложете на последно място и се фокусирайте върху проверките!

## Демонстрация: писане тестове на живо

Кодът за който ще се пишат unit тестове са намира във файлa `live-demo/bank.py` и изглежда ето така:

```python
class BankAccount(object):
    def __init__(self, name, balance):
        self.name = name
        self._balance = balance
        self._history = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError('Deposit amount must be positive!')

        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError('Withdraw amount must be positive!')

        if amount <= self._balance:
            self._balance -= amount
            return True
        else:
            self._history.append("Withdraw for %d failed" % amount)

        return False

    def transfer_to(self, other_account, how_much):
        self.withdraw(how_much)
        other_account.deposit(how_much)

```

... TODO: писане тестове на живо и илюстриране на теорията обяснена в предишните параграфи ...